# Notebook 02 — Supply Network Data

## Objective
Design a hypothetical hardware supply network
capable of satisfying the 12-week demand plan
developed in Notebook 01.

## Network Structure
- 1 reference hardware product
- 3 simplified component groups
- 2 component suppliers
- 1 manufacturing plant
- 2 distribution centers
- 2 regional customer markets
- 2 transportation modes

## Business Decisions
1. How many components should we purchase?
2. Which suppliers should fulfill requirements?
3. How many units should we manufacture weekly?
4. How should inventory be allocated between DCs?
5. When should expedited transportation be used?
6. How can we minimize cost while maintaining service?

## Data Integrity
All network costs, capacities, and lead times
will initially be documented hypothetical assumptions.


In [1]:
# STEP 1: Load demand planning data

import pandas as pd
from pathlib import Path

# Locate repository root
repo_root = Path.cwd()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# Load demand plan
demand_path = (
    repo_root
    / "data"
    / "processed"
    / "demand_plan_12_weeks.csv"
)

assert demand_path.is_file(), (
    f"Demand file not found: {demand_path}"
)

demand = pd.read_csv(
    demand_path,
    parse_dates=["Week_Start"]
)

# Validate imported demand
assert len(demand) == 12
assert demand["Week"].tolist() == list(range(1, 13))
assert demand["Week_Start"].is_unique
assert demand["Demand_Units"].notna().all()
assert (demand["Demand_Units"] >= 0).all()
assert demand["Demand_Units"].sum() == 6180

print("Notebook 02 initialized successfully!")
print("Planning weeks:", len(demand))
print("Total demand:", demand["Demand_Units"].sum())
print("Product:", demand["SKU"].unique())

display(demand)


Notebook 02 initialized successfully!
Planning weeks: 12
Total demand: 6180.0
Product: <StringArray>
['AIRPODS_REFERENCE']
Length: 1, dtype: str


,Week,Week_Start,SKU,Demand_Units,Demand_Source,Scenario
0,1,2020-01-05,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
1,2,2020-01-12,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
2,3,2020-01-19,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
3,4,2020-01-26,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
4,5,2020-02-02,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
5,6,2020-02-09,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
6,7,2020-02-16,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
7,8,2020-02-23,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
8,9,2020-03-01,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
9,10,2020-03-08,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline


## Step 2 — Hardware Bill of Materials

### Objective
Define the components required to assemble one
AIRPODS_REFERENCE finished product.

### Planning BOM
- Electronics Assembly: 1 per finished unit
- Battery Module: 1 per finished unit
- Enclosure & Packaging Kit: 1 per finished unit

### Assumptions
This is a simplified, hypothetical planning BOM.
It does not represent Apple's actual product design.

Component quantities will be used to calculate
material requirements and constrain production.

In [2]:
# STEP 2: Build hardware Bill of Materials

import pandas as pd

bom = pd.DataFrame({
    "SKU": [
        "AIRPODS_REFERENCE",
        "AIRPODS_REFERENCE",
        "AIRPODS_REFERENCE"
    ],
    "Component_ID": [
        "ELEC_001",
        "BATT_001",
        "ENCL_001"
    ],
    "Component_Name": [
        "Electronics Assembly",
        "Battery Module",
        "Enclosure and Packaging Kit"
    ],
    "Units_Per_Finished_Unit": [1, 1, 1],
    "Unit_Of_Measure": ["Each", "Each", "Kit"],
    "Data_Source": [
        "Hypothetical_Assumption"
    ] * 3
})

# Calculate total component requirements
total_finished_demand = demand["Demand_Units"].sum()

bom["Total_12wk_Requirement"] = (
    bom["Units_Per_Finished_Unit"]
    * total_finished_demand
)

print("Hardware BOM created successfully!")
print("Finished product demand:", total_finished_demand)
print("Number of component groups:", len(bom))

display(bom)


Hardware BOM created successfully!
Finished product demand: 6180.0
Number of component groups: 3


,SKU,Component_ID,Component_Name,Units_Per_Finished_Unit,Unit_Of_Measure,Data_Source,Total_12wk_Requirement
0,AIRPODS_REFERENCE,ELEC_001,Electronics Assembly,1,Each,Hypothetical_Assumption,6180.0
1,AIRPODS_REFERENCE,BATT_001,Battery Module,1,Each,Hypothetical_Assumption,6180.0
2,AIRPODS_REFERENCE,ENCL_001,Enclosure and Packaging Kit,1,Kit,Hypothetical_Assumption,6180.0


In [3]:
# STEP 2D: Validate and export BOM

# Validate BOM
assert len(bom) == 3
assert bom["Component_ID"].is_unique
assert bom["SKU"].eq("AIRPODS_REFERENCE").all()
assert (bom["Units_Per_Finished_Unit"] > 0).all()
assert bom["Component_ID"].notna().all()

# Save only master BOM attributes
bom_master = bom.drop(
    columns=["Total_12wk_Requirement"]
)

output_dir = repo_root / "data" / "assumptions"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "bom.csv"

bom_master.to_csv(output_file, index=False)

print("SUCCESS: Hardware BOM exported")
print("File:", output_file)
print("Components:", len(bom_master))
print("Validation checks: PASSED")


SUCCESS: Hardware BOM exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/bom.csv
Components: 3
Validation checks: PASSED


## Step 3 — Supplier Network Design

### Objective
Define two hypothetical component suppliers with
different procurement costs, weekly capacities,
and purchasing lead times.

### Supplier Strategy
- Supplier A: Lower cost, longer lead time
- Supplier B: Higher cost, shorter lead time

### Planning Assumptions
Both suppliers can provide all three component groups.

Supplier capacity is specified separately for
each component and each week.

Supplier costs and lead times are hypothetical,
not actual company or market quotations.

### Optimization Objective
Determine how much to purchase from each supplier
while balancing procurement cost, availability,
lead times, and manufacturing requirements.

In [4]:
# STEP 3: Build supplier network

import pandas as pd

suppliers = pd.DataFrame([
    # Supplier A - lower-cost primary source
    ["SUP_A", "Primary Supplier", "ELEC_001", 32, 450, 2],
    ["SUP_A", "Primary Supplier", "BATT_001", 14, 450, 2],
    ["SUP_A", "Primary Supplier", "ENCL_001", 8, 450, 2],

    # Supplier B - higher-cost backup source
    ["SUP_B", "Backup Supplier", "ELEC_001", 40, 250, 1],
    ["SUP_B", "Backup Supplier", "BATT_001", 18, 250, 1],
    ["SUP_B", "Backup Supplier", "ENCL_001", 11, 250, 1],
], columns=[
    "Supplier_ID",
    "Supplier_Name",
    "Component_ID",
    "Unit_Cost_USD",
    "Weekly_Capacity_Units",
    "Lead_Time_Weeks"
])

suppliers["Eligible"] = True
suppliers["Data_Source"] = "Hypothetical_Assumption"

print("Supplier network created successfully!")
print("Suppliers:", suppliers["Supplier_ID"].nunique())
print("Supplier-component options:", len(suppliers))

display(suppliers)


Supplier network created successfully!
Suppliers: 2
Supplier-component options: 6


,Supplier_ID,Supplier_Name,Component_ID,Unit_Cost_USD,Weekly_Capacity_Units,Lead_Time_Weeks,Eligible,Data_Source
0,SUP_A,Primary Supplier,ELEC_001,32,450,2,True,Hypothetical_Assumption
1,SUP_A,Primary Supplier,BATT_001,14,450,2,True,Hypothetical_Assumption
2,SUP_A,Primary Supplier,ENCL_001,8,450,2,True,Hypothetical_Assumption
3,SUP_B,Backup Supplier,ELEC_001,40,250,1,True,Hypothetical_Assumption
4,SUP_B,Backup Supplier,BATT_001,18,250,1,True,Hypothetical_Assumption
5,SUP_B,Backup Supplier,ENCL_001,11,250,1,True,Hypothetical_Assumption


In [5]:
# STEP 3D: Validate and export supplier network

from pathlib import Path

# Validate supplier data
assert len(suppliers) == 6
assert suppliers["Supplier_ID"].nunique() == 2

assert not suppliers.duplicated(
    subset=["Supplier_ID", "Component_ID"]
).any()

assert set(suppliers["Component_ID"]) == set(
    bom["Component_ID"]
)

assert suppliers["Unit_Cost_USD"].gt(0).all()
assert suppliers["Weekly_Capacity_Units"].gt(0).all()
assert suppliers["Lead_Time_Weeks"].ge(0).all()
assert suppliers["Eligible"].all()
assert suppliers.notna().all().all()

# Check sourcing coverage for weekly gross demand
weekly_requirements = (
    bom.set_index("Component_ID")[
        "Units_Per_Finished_Unit"
    ] * demand["Demand_Units"].max()
)

weekly_capacity = suppliers.groupby(
    "Component_ID"
)["Weekly_Capacity_Units"].sum()

capacity_check = pd.DataFrame({
    "Weekly_Requirement": weekly_requirements,
    "Combined_Supplier_Capacity": weekly_capacity
})

capacity_check["Capacity_Sufficient"] = (
    capacity_check["Combined_Supplier_Capacity"]
    >= capacity_check["Weekly_Requirement"]
)

assert capacity_check["Capacity_Sufficient"].all()

display(capacity_check)

# Export supplier master
output_dir = repo_root / "data" / "assumptions"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "suppliers.csv"

suppliers.to_csv(output_file, index=False)

print("SUCCESS: Supplier network exported")
print("File:", output_file)
print("Supplier options:", len(suppliers))
print("Validation checks: PASSED")

,Weekly_Requirement,Combined_Supplier_Capacity,Capacity_Sufficient
Component_ID,,,
BATT_001,515.0,700,True
ELEC_001,515.0,700,True
ENCL_001,515.0,700,True


SUCCESS: Supplier network exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/suppliers.csv
Supplier options: 6
Validation checks: PASSED


## Step 4 — Manufacturing Plant Capacity

### Objective
Define a hypothetical assembly plant that converts
purchased components into finished hardware products.

### Manufacturing Assumptions
- Plant: Central Assembly Plant
- Product: AIRPODS_REFERENCE
- Weekly production capacity: 580 units
- Assembly cost: $22 per finished unit
- Production lead time: 0 weeks

### Planning Logic
Manufacturing consumes components according to the BOM.

The production lead time of zero means assembly
can be completed within the same weekly planning
period, provided components are available.

Production cannot exceed:
1. Available manufacturing capacity
2. Available component quantities

### Future Optimization
The optimizer will determine weekly production
quantities based on demand, supplier deliveries,
inventory availability, and total network cost.

All manufacturing parameters are hypothetical.


In [6]:
# STEP 4: Build manufacturing plant master

import pandas as pd

plants = pd.DataFrame({
    "Plant_ID": ["PLANT_01"],
    "Plant_Name": ["Central Assembly Plant"],
    "SKU": ["AIRPODS_REFERENCE"],
    "Weekly_Capacity_Units": [580],
    "Assembly_Cost_Per_Unit_USD": [22],
    "Production_Lead_Time_Weeks": [0],
    "Data_Source": ["Hypothetical_Assumption"]
})

# Calculate planning metrics
weekly_demand = demand["Demand_Units"].mean()
plant_capacity = plants["Weekly_Capacity_Units"].iloc[0]

required_utilization = (
    weekly_demand / plant_capacity * 100
)

weekly_spare_capacity = (
    plant_capacity - weekly_demand
)

total_capacity = plant_capacity * len(demand)

print("Manufacturing plant created successfully!")
print("Plants:", len(plants))
print("Weekly demand:", weekly_demand)
print("Weekly capacity:", plant_capacity)
print("Required utilization:", round(required_utilization, 2), "%")
print("Weekly spare capacity:", weekly_spare_capacity)
print("12-week capacity:", total_capacity)

display(plants)


Manufacturing plant created successfully!
Plants: 1
Weekly demand: 515.0
Weekly capacity: 580
Required utilization: 88.79 %
Weekly spare capacity: 65.0
12-week capacity: 6960


,Plant_ID,Plant_Name,SKU,Weekly_Capacity_Units,Assembly_Cost_Per_Unit_USD,Production_Lead_Time_Weeks,Data_Source
0,PLANT_01,Central Assembly Plant,AIRPODS_REFERENCE,580,22,0,Hypothetical_Assumption


In [7]:
# STEP 4a: Validate and export manufacturing plant

from pathlib import Path

# Validate manufacturing records
assert len(plants) == 1
assert plants["Plant_ID"].is_unique

assert set(plants["SKU"]).issubset(
    set(demand["SKU"])
)

assert plants["Weekly_Capacity_Units"].gt(0).all()
assert plants["Assembly_Cost_Per_Unit_USD"].ge(0).all()
assert plants["Production_Lead_Time_Weeks"].ge(0).all()

assert plants.notna().all().all()

# Capacity check
assert (
    plants["Weekly_Capacity_Units"].sum()
    >= demand["Demand_Units"].max()
)

# Export plant master
output_dir = repo_root / "data" / "assumptions"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "plants.csv"

plants.to_csv(output_file, index=False)

print("SUCCESS: Manufacturing plant exported")
print("File:", output_file)
print("Plants:", len(plants))
print("Validation checks: PASSED")


SUCCESS: Manufacturing plant exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/plants.csv
Plants: 1
Validation checks: PASSED


## Step 5 — Distribution Center Design

### Objective
Define two regional distribution centers for
finished hardware products.

### Distribution Network
- East DC: Eastern customer market
- West DC: Western customer market

### Storage Capacity
- East DC: 450 units
- West DC: 400 units

### Weekly Inventory Holding Costs
- East DC: $1.25 per unit
- West DC: $1.40 per unit

### Planning Logic
Finished goods are manufactured at PLANT_01
and transported to regional distribution centers.

Inventory is tracked weekly.

The optimizer must ensure that inventory on hand
does not exceed each distribution center's
storage capacity.

Regional demand allocation, shipment timing,
transportation costs, and beginning inventory
will be defined in subsequent steps.

### Data Integrity
All capacities and costs are hypothetical
planning assumptions, not actual company data.


In [9]:
# STEP 5: Build distribution center master

import pandas as pd

distribution_centers = pd.DataFrame({
    "DC_ID": [
        "DC_EAST",
        "DC_WEST"
    ],
    "DC_Name": [
        "East Distribution Center",
        "West Distribution Center"
    ],
    "Region": [
        "East",
        "West"
    ],
    "SKU": [
        "AIRPODS_REFERENCE",
        "AIRPODS_REFERENCE"
    ],
    "Storage_Capacity_Units": [
        450,
        400
    ],
    "Holding_Cost_Per_Unit_Per_Week_USD": [
        1.25,
        1.40
    ],
    "Data_Source": [
        "Hypothetical_Assumption",
        "Hypothetical_Assumption"
    ]
})

# Calculate network-wide reference metrics
total_storage_capacity = (
    distribution_centers["Storage_Capacity_Units"].sum()
)

weekly_demand = demand["Demand_Units"].mean()

capacity_to_weekly_demand = (
    total_storage_capacity / weekly_demand
)

print("Distribution center network created!")
print("Number of DCs:", len(distribution_centers))
print("Total storage capacity:", total_storage_capacity)
print("Average weekly demand:", weekly_demand)
print(
    "Storage capacity / weekly demand:",
    round(capacity_to_weekly_demand, 2)
)

display(distribution_centers)


Distribution center network created!
Number of DCs: 2
Total storage capacity: 850
Average weekly demand: 515.0
Storage capacity / weekly demand: 1.65


,DC_ID,DC_Name,Region,SKU,Storage_Capacity_Units,Holding_Cost_Per_Unit_Per_Week_USD,Data_Source
0,DC_EAST,East Distribution Center,East,AIRPODS_REFERENCE,450,1.25,Hypothetical_Assumption
1,DC_WEST,West Distribution Center,West,AIRPODS_REFERENCE,400,1.40,Hypothetical_Assumption


In [10]:
# STEP 5a: Validate and export distribution centers

from pathlib import Path

# Validate distribution center records
assert len(distribution_centers) == 2

assert distribution_centers["DC_ID"].is_unique

assert set(distribution_centers["Region"]) == {
    "East", "West"
}

assert set(distribution_centers["SKU"]).issubset(
    set(demand["SKU"])
)

assert distribution_centers[
    "Storage_Capacity_Units"
].gt(0).all()

assert distribution_centers[
    "Holding_Cost_Per_Unit_Per_Week_USD"
].ge(0).all()

assert distribution_centers.notna().all().all()

# Validate total storage versus average weekly demand
assert (
    distribution_centers["Storage_Capacity_Units"].sum()
    >= demand["Demand_Units"].mean()
)

# Export DC master
output_dir = repo_root / "data" / "assumptions"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir / "distribution_centers.csv"
)

distribution_centers.to_csv(
    output_file,
    index=False
)

print("SUCCESS: Distribution centers exported")
print("File:", output_file)
print("Distribution centers:", len(distribution_centers))
print("Validation checks: PASSED")


SUCCESS: Distribution centers exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/distribution_centers.csv
Distribution centers: 2
Validation checks: PASSED



## Step 6 — Transportation Network Design

### Objective
Define transportation options connecting the
manufacturing plant to regional distribution centers.

### Transportation Modes
1. Standard: Lower cost, longer transit time
2. Expedited: Higher cost, shorter transit time

### Network
- PLANT_01 to DC_EAST
- PLANT_01 to DC_WEST

Each route supports both transportation modes.

### Assumptions
- Freight costs are USD per finished product shipped.
- Transit times are measured in whole planning weeks.
- Zero-week transit means same-week delivery.
- Weekly lane capacities limit shipment quantities.
- Supplier-to-plant inbound transportation is
  assumed included in supplier purchase costs.
- All transportation values are hypothetical.

### Optimization Decisions
The future model will determine:
- Units shipped along each lane
- Transportation mode selection
- Weekly shipment timing
- Inventory arrival timing
- Freight costs versus customer service trade-offs


In [11]:
# STEP 6: Build transportation network

import pandas as pd

transport_lanes = pd.DataFrame([
    # East distribution center
    ["LANE_001", "PLANT_01", "DC_EAST",
     "Standard", 3.50, 1, 350],

    ["LANE_002", "PLANT_01", "DC_EAST",
     "Expedited", 7.50, 0, 250],

    # West distribution center
    ["LANE_003", "PLANT_01", "DC_WEST",
     "Standard", 5.00, 2, 300],

    ["LANE_004", "PLANT_01", "DC_WEST",
     "Expedited", 10.00, 1, 250],

], columns=[
    "Lane_ID",
    "Origin_Plant_ID",
    "Destination_DC_ID",
    "Transport_Mode",
    "Cost_Per_Unit_USD",
    "Transit_Time_Weeks",
    "Weekly_Lane_Capacity_Units"
])

transport_lanes["SKU"] = "AIRPODS_REFERENCE"

transport_lanes["Data_Source"] = (
    "Hypothetical_Assumption"
)

print("Transportation network created successfully!")
print("Transportation lanes:", len(transport_lanes))
print(
    "Distribution centers:",
    transport_lanes["Destination_DC_ID"].nunique()
)
print(
    "Shipping modes:",
    transport_lanes["Transport_Mode"].nunique()
)

display(transport_lanes)


Transportation network created successfully!
Transportation lanes: 4
Distribution centers: 2
Shipping modes: 2


,Lane_ID,Origin_Plant_ID,Destination_DC_ID,Transport_Mode,Cost_Per_Unit_USD,Transit_Time_Weeks,Weekly_Lane_Capacity_Units,SKU,Data_Source
0,LANE_001,PLANT_01,DC_EAST,Standard,3.5,1,350,AIRPODS_REFERENCE,Hypothetical_Assumption
1,LANE_002,PLANT_01,DC_EAST,Expedited,7.5,0,250,AIRPODS_REFERENCE,Hypothetical_Assumption
2,LANE_003,PLANT_01,DC_WEST,Standard,5.0,2,300,AIRPODS_REFERENCE,Hypothetical_Assumption
3,LANE_004,PLANT_01,DC_WEST,Expedited,10.0,1,250,AIRPODS_REFERENCE,Hypothetical_Assumption


In [12]:

# STEP 6D: Compare standard versus expedited freight

comparison = transport_lanes.pivot(
    index="Destination_DC_ID",
    columns="Transport_Mode",
    values="Cost_Per_Unit_USD"
)

comparison["Expedited_Premium_USD"] = (
    comparison["Expedited"]
    - comparison["Standard"]
)

comparison["Expedited_Premium_Pct"] = (
    comparison["Expedited_Premium_USD"]
    / comparison["Standard"]
    * 100
)

print("Transportation cost comparison:")
display(comparison.round(2))


Transportation cost comparison:


Transport_Mode,Expedited,Standard,Expedited_Premium_USD,Expedited_Premium_Pct
Destination_DC_ID,,,,
DC_EAST,7.5,3.5,4.0,114.29
DC_WEST,10.0,5.0,5.0,100.00


In [15]:
# STEP 6b: Validate and export transportation network

from pathlib import Path

# Validate basic data integrity
assert len(transport_lanes) == 4

assert transport_lanes["Lane_ID"].is_unique

assert not transport_lanes.duplicated(
    subset=[
        "Origin_Plant_ID",
        "Destination_DC_ID",
        "Transport_Mode",
        "SKU"
    ]
).any()

# Validate references to plants and DCs
assert set(transport_lanes["Origin_Plant_ID"]).issubset(
    set(plants["Plant_ID"])
)

assert set(transport_lanes["Destination_DC_ID"]).issubset(
    set(distribution_centers["DC_ID"])
)

assert set(transport_lanes["SKU"]).issubset(
    set(demand["SKU"])
)

# Validate numerical parameters
assert transport_lanes[
    "Cost_Per_Unit_USD"
].ge(0).all()

assert transport_lanes[
    "Transit_Time_Weeks"
].ge(0).all()

assert transport_lanes[
    "Weekly_Lane_Capacity_Units"
].gt(0).all()

assert transport_lanes.notna().all().all()

# Every destination must have both shipping modes
expected_modes = {"Standard", "Expedited"}

for dc_id, group in transport_lanes.groupby(
    "Destination_DC_ID"
):
    assert set(group["Transport_Mode"]) == expected_modes

    standard = group.loc[
        group["Transport_Mode"] == "Standard"
    ].iloc[0]

    expedited = group.loc[
        group["Transport_Mode"] == "Expedited"
    ].iloc[0]

    assert (
        expedited["Cost_Per_Unit_USD"]
        > standard["Cost_Per_Unit_USD"]
    )

    assert (
        expedited["Transit_Time_Weeks"]
        < standard["Transit_Time_Weeks"]
    )

# Export transportation network
output_dir = repo_root / "data" / "assumptions"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "transport_lanes.csv"

transport_lanes.to_csv(output_file, index=False)

print("SUCCESS: Transportation network exported")
print("File:", output_file)
print("Transportation lanes:", len(transport_lanes))
print("Validation checks: PASSED")


SUCCESS: Transportation network exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/transport_lanes.csv
Transportation lanes: 4
Validation checks: PASSED


## Step 7 — Initial Inventory Design

### Objective
Define beginning inventory across the manufacturing
plant and distribution centers.

### Component Inventory
Each of the three component groups begins with
1,100 units at PLANT_01.

This provides approximately two weeks of
component coverage at baseline production demand.

### Finished Goods Inventory
- Manufacturing Plant: 0 units
- East DC: 330 units
- West DC: 230 units

### Inventory Timing
Beginning inventory is available at the start
of Week 1, before production and customer demand.

No pre-horizon supplier deliveries or in-transit
shipments are assumed.

### Inventory Rules
- Production consumes components according to the BOM.
- Production creates finished goods inventory.
- Shipments transfer finished goods from plant to DCs.
- Customer fulfillment consumes DC inventory.
- Inventory balances must remain nonnegative.
- DC inventory must respect storage limits.

### Data Integrity
All beginning inventory quantities are hypothetical.

Beginning inventory will be explicitly included in
future inventory-balance equations, preventing
the model from creating or double-counting stock.


In [16]:
# STEP 7: Build initial inventory master

import pandas as pd

initial_inventory = pd.DataFrame([
    # Components stored at manufacturing plant
    ["PLANT_01", "Component", "ELEC_001", 1100, "Each"],
    ["PLANT_01", "Component", "BATT_001", 1100, "Each"],
    ["PLANT_01", "Component", "ENCL_001", 1100, "Kit"],

    # Finished goods at manufacturing plant
    ["PLANT_01", "Finished_Good",
     "AIRPODS_REFERENCE", 0, "Each"],

    # Finished goods at distribution centers
    ["DC_EAST", "Finished_Good",
     "AIRPODS_REFERENCE", 330, "Each"],

    ["DC_WEST", "Finished_Good",
     "AIRPODS_REFERENCE", 230, "Each"],

], columns=[
    "Location_ID",
    "Item_Type",
    "Item_ID",
    "Beginning_Inventory_Units",
    "Unit_Of_Measure"
])

initial_inventory["Data_Source"] = (
    "Hypothetical_Assumption"
)

print("Initial inventory created successfully!")
print("Inventory records:", len(initial_inventory))
print(
    "Locations:",
    initial_inventory["Location_ID"].nunique()
)

display(initial_inventory)


Initial inventory created successfully!
Inventory records: 6
Locations: 3


,Location_ID,Item_Type,Item_ID,Beginning_Inventory_Units,Unit_Of_Measure,Data_Source
0,PLANT_01,Component,ELEC_001,1100,Each,Hypothetical_Assumption
1,PLANT_01,Component,BATT_001,1100,Each,Hypothetical_Assumption
2,PLANT_01,Component,ENCL_001,1100,Kit,Hypothetical_Assumption
3,PLANT_01,Finished_Good,AIRPODS_REFERENCE,0,Each,Hypothetical_Assumption
4,DC_EAST,Finished_Good,AIRPODS_REFERENCE,330,Each,Hypothetical_Assumption
5,DC_WEST,Finished_Good,AIRPODS_REFERENCE,230,Each,Hypothetical_Assumption


In [17]:
# STEP 7: Analyze beginning inventory coverage

weekly_demand = demand["Demand_Units"].mean()

# Component inventory
component_inventory = initial_inventory[
    initial_inventory["Item_Type"] == "Component"
].copy()

component_coverage = component_inventory.merge(
    bom[[
        "Component_ID",
        "Units_Per_Finished_Unit"
    ]],
    left_on="Item_ID",
    right_on="Component_ID",
    how="left",
    validate="one_to_one"
)

component_coverage["Weekly_Component_Requirement"] = (
    component_coverage["Units_Per_Finished_Unit"]
    * weekly_demand
)

component_coverage["Weeks_Of_Coverage"] = (
    component_coverage["Beginning_Inventory_Units"]
    / component_coverage["Weekly_Component_Requirement"]
)

print("COMPONENT INVENTORY COVERAGE:")
display(component_coverage[[
    "Item_ID",
    "Beginning_Inventory_Units",
    "Weekly_Component_Requirement",
    "Weeks_Of_Coverage"
]].round(2))

# Finished goods inventory
finished_inventory = initial_inventory[
    initial_inventory["Item_Type"] == "Finished_Good"
]

total_finished_inventory = (
    finished_inventory["Beginning_Inventory_Units"].sum()
)

finished_goods_coverage = (
    total_finished_inventory / weekly_demand
)

print("\nFINISHED GOODS INVENTORY:")
print("Starting finished goods:", total_finished_inventory)
print("Weekly customer demand:", weekly_demand)
print(
    "Aggregate weeks of finished goods coverage:",
    round(finished_goods_coverage, 2)
)


COMPONENT INVENTORY COVERAGE:


,Item_ID,Beginning_Inventory_Units,Weekly_Component_Requirement,Weeks_Of_Coverage
0,ELEC_001,1100,515.0,2.14
1,BATT_001,1100,515.0,2.14
2,ENCL_001,1100,515.0,2.14



FINISHED GOODS INVENTORY:
Starting finished goods: 560
Weekly customer demand: 515.0
Aggregate weeks of finished goods coverage: 1.09


In [18]:
# STEP 7a: Validate and export initial inventory

from pathlib import Path

# Basic integrity checks
assert len(initial_inventory) == 6

assert not initial_inventory.duplicated(
    subset=["Location_ID", "Item_Type", "Item_ID"]
).any()

assert initial_inventory[
    "Beginning_Inventory_Units"
].ge(0).all()

assert initial_inventory.notna().all().all()

# Validate item identifiers
valid_components = set(bom["Component_ID"])
valid_products = set(demand["SKU"])

component_rows = initial_inventory[
    initial_inventory["Item_Type"] == "Component"
]

finished_rows = initial_inventory[
    initial_inventory["Item_Type"] == "Finished_Good"
]

assert set(component_rows["Item_ID"]) == valid_components

assert set(finished_rows["Item_ID"]).issubset(
    valid_products
)

# Validate inventory locations
valid_locations = (
    set(plants["Plant_ID"])
    | set(distribution_centers["DC_ID"])
)

assert set(initial_inventory["Location_ID"]).issubset(
    valid_locations
)

# Components must be at the manufacturing plant
assert set(component_rows["Location_ID"]) == {
    "PLANT_01"
}

# Check starting DC inventory against storage capacity
dc_inventory = finished_rows[
    finished_rows["Location_ID"].isin(
        distribution_centers["DC_ID"]
    )
]

dc_check = distribution_centers[[
    "DC_ID",
    "Storage_Capacity_Units"
]].merge(
    dc_inventory[[
        "Location_ID",
        "Beginning_Inventory_Units"
    ]],
    left_on="DC_ID",
    right_on="Location_ID",
    how="left",
    validate="one_to_one"
)

assert dc_check["Beginning_Inventory_Units"].notna().all()

dc_check["Within_Capacity"] = (
    dc_check["Beginning_Inventory_Units"]
    <= dc_check["Storage_Capacity_Units"]
)

assert dc_check["Within_Capacity"].all()

print("DISTRIBUTION CENTER CAPACITY CHECK:")
display(dc_check[[
    "DC_ID",
    "Beginning_Inventory_Units",
    "Storage_Capacity_Units",
    "Within_Capacity"
]])

# Verify two-week initial component coverage
assert (
    component_coverage["Weeks_Of_Coverage"] >= 2
).all()

# Export initial inventory
output_dir = repo_root / "data" / "assumptions"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "initial_inventory.csv"

initial_inventory.to_csv(output_file, index=False)

print("\nSUCCESS: Initial inventory exported")
print("File:", output_file)
print("Inventory records:", len(initial_inventory))
print("Validation checks: PASSED")


DISTRIBUTION CENTER CAPACITY CHECK:


,DC_ID,Beginning_Inventory_Units,Storage_Capacity_Units,Within_Capacity
0,DC_EAST,330,450,True
1,DC_WEST,230,400,True



SUCCESS: Initial inventory exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/initial_inventory.csv
Inventory records: 6
Validation checks: PASSED



## Step 8 — Regional Demand Allocation

### Objective
Allocate Project 1's weekly hardware demand
between two regional customer markets.

### Regional Assumptions
- East Region: 60% of demand
- West Region: 40% of demand

### Distribution Center Mapping
- East Region → DC_EAST
- West Region → DC_WEST

### Weekly Baseline Demand
- Total: 515 units
- East: 309 units
- West: 206 units

### Planning Logic
Each region is served by its assigned
distribution center.

The optimizer will determine how much inventory
to ship to each distribution center to satisfy
regional customer demand.

Unfulfilled regional demand will be explicitly
tracked in the optimization model.

### Data Integrity
Regional allocation percentages are hypothetical.

The original four-week Project 1 forecast and
the eight-week planning extension retain their
source labels.

Regional demand must reconcile exactly to
the original 12-week demand plan.


In [19]:
# STEP 8: Build regional demand allocation master

import pandas as pd

market_allocation = pd.DataFrame({
    "Region": [
        "East",
        "West"
    ],
    "DC_ID": [
        "DC_EAST",
        "DC_WEST"
    ],
    "Demand_Share": [
        0.60,
        0.40
    ],
    "Data_Source": [
        "Hypothetical_Assumption",
        "Hypothetical_Assumption"
    ]
})

# Validate allocation percentages
assert abs(
    market_allocation["Demand_Share"].sum() - 1.0
) < 1e-9

assert market_allocation["Region"].is_unique
assert market_allocation["DC_ID"].is_unique

# Validate distribution center references
assert set(market_allocation["DC_ID"]) == set(
    distribution_centers["DC_ID"]
)

region_dc_check = market_allocation.merge(
    distribution_centers[["DC_ID", "Region"]],
    on="DC_ID",
    suffixes=("_Allocation", "_DC"),
    validate="one_to_one"
)

assert (
    region_dc_check["Region_Allocation"]
    == region_dc_check["Region_DC"]
).all()

print("Market allocation created successfully!")
print("Regions:", len(market_allocation))
print(
    "Total allocation:",
    market_allocation["Demand_Share"].sum()
)

display(market_allocation)


Market allocation created successfully!
Regions: 2
Total allocation: 1.0


,Region,DC_ID,Demand_Share,Data_Source
0,East,DC_EAST,0.6,Hypothetical_Assumption
1,West,DC_WEST,0.4,Hypothetical_Assumption


In [20]:
# STEP 8a: Generate regional weekly demand

import numpy as np

# Combine weekly demand with regional allocation
regional_demand = demand.merge(
    market_allocation,
    how="cross"
)

# Calculate regional demand
regional_demand["Regional_Demand_Units"] = (
    regional_demand["Demand_Units"]
    * regional_demand["Demand_Share"]
)

# V1 uses whole finished-product units.
# Confirm the 60/40 split produces whole units.
assert np.allclose(
    regional_demand["Regional_Demand_Units"],
    regional_demand["Regional_Demand_Units"].round()
), "Regional demand requires an integer allocation rule"

regional_demand["Regional_Demand_Units"] = (
    regional_demand["Regional_Demand_Units"]
    .round()
    .astype(int)
)

# Replace aggregate demand with regional demand
regional_demand = regional_demand.drop(
    columns=["Demand_Units"]
).rename(
    columns={
        "Regional_Demand_Units": "Demand_Units"
    }
)

# Arrange columns
regional_demand = regional_demand[[
    "Week",
    "Week_Start",
    "SKU",
    "Region",
    "DC_ID",
    "Demand_Share",
    "Demand_Units",
    "Demand_Source",
    "Scenario",
    "Data_Source"
]]

# Sort records
regional_demand = regional_demand.sort_values(
    ["Week", "Region"]
).reset_index(drop=True)

print("Regional demand dataset created!")
print("Total records:", len(regional_demand))
print("Regions:", regional_demand["Region"].nunique())

display(regional_demand.head(10))


Regional demand dataset created!
Total records: 24
Regions: 2


,Week,Week_Start,SKU,Region,DC_ID,Demand_Share,Demand_Units,Demand_Source,Scenario,Data_Source
0,1,2020-01-05,AIRPODS_REFERENCE,East,DC_EAST,0.6,309,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
1,1,2020-01-05,AIRPODS_REFERENCE,West,DC_WEST,0.4,206,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
2,2,2020-01-12,AIRPODS_REFERENCE,East,DC_EAST,0.6,309,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
3,2,2020-01-12,AIRPODS_REFERENCE,West,DC_WEST,0.4,206,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
4,3,2020-01-19,AIRPODS_REFERENCE,East,DC_EAST,0.6,309,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
5,3,2020-01-19,AIRPODS_REFERENCE,West,DC_WEST,0.4,206,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
6,4,2020-01-26,AIRPODS_REFERENCE,East,DC_EAST,0.6,309,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
7,4,2020-01-26,AIRPODS_REFERENCE,West,DC_WEST,0.4,206,Project1_Original_Forecast,Baseline,Hypothetical_Assumption
8,5,2020-02-02,AIRPODS_REFERENCE,East,DC_EAST,0.6,309,Flat_Extension_Assumption,Baseline,Hypothetical_Assumption
9,5,2020-02-02,AIRPODS_REFERENCE,West,DC_WEST,0.4,206,Flat_Extension_Assumption,Baseline,Hypothetical_Assumption


In [21]:
# STEP 8b: Validate regional demand reconciliation

# Calculate regional totals
regional_summary = regional_demand.groupby(
    ["Region", "DC_ID"],
    as_index=False
)["Demand_Units"].sum()

print("12-WEEK REGIONAL DEMAND SUMMARY:")
display(regional_summary)

# Verify overall demand
original_total = demand["Demand_Units"].sum()
regional_total = regional_demand["Demand_Units"].sum()

assert regional_total == original_total

# Verify expected number of records
assert len(regional_demand) == 24

# Verify regional totals
east_total = regional_summary.loc[
    regional_summary["Region"] == "East",
    "Demand_Units"
].iloc[0]

west_total = regional_summary.loc[
    regional_summary["Region"] == "West",
    "Demand_Units"
].iloc[0]

assert east_total == 3708
assert west_total == 2472

# Verify every week reconciles to Project 1
weekly_check = regional_demand.groupby(
    "Week",
    as_index=False
)["Demand_Units"].sum()

weekly_check = weekly_check.merge(
    demand[["Week", "Demand_Units"]],
    on="Week",
    suffixes=("_Regional", "_Original"),
    validate="one_to_one"
)

weekly_check["Difference"] = (
    weekly_check["Demand_Units_Regional"]
    - weekly_check["Demand_Units_Original"]
)

assert weekly_check["Difference"].eq(0).all()

# Check uniqueness and completeness
assert not regional_demand.duplicated(
    subset=["Week", "SKU", "Region"]
).any()

assert regional_demand.notna().all().all()
assert regional_demand["Demand_Units"].ge(0).all()

assert regional_demand.groupby("Week")[
    "Region"
].nunique().eq(2).all()

print("\nWEEKLY DEMAND RECONCILIATION:")
display(weekly_check)

print("\nVALIDATION SUMMARY")
print("Original demand:", original_total)
print("Regional demand:", regional_total)
print("Difference:", regional_total - original_total)
print("Validation checks: PASSED")

12-WEEK REGIONAL DEMAND SUMMARY:


,Region,DC_ID,Demand_Units
0,East,DC_EAST,3708
1,West,DC_WEST,2472



WEEKLY DEMAND RECONCILIATION:


,Week,Demand_Units_Regional,Demand_Units_Original,Difference
0,1,515,515.0,0.0
1,2,515,515.0,0.0
2,3,515,515.0,0.0
3,4,515,515.0,0.0
4,5,515,515.0,0.0
5,6,515,515.0,0.0
6,7,515,515.0,0.0
7,8,515,515.0,0.0
8,9,515,515.0,0.0
9,10,515,515.0,0.0



VALIDATION SUMMARY
Original demand: 6180.0
Regional demand: 6180
Difference: 0.0
Validation checks: PASSED


In [22]:

# STEP 8c: Export regional planning datasets

from pathlib import Path

# Output directories
assumptions_dir = repo_root / "data" / "assumptions"
processed_dir = repo_root / "data" / "processed"

assumptions_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# Market allocation assumptions
allocation_file = (
    assumptions_dir / "market_allocation.csv"
)

market_allocation.to_csv(
    allocation_file,
    index=False
)

# Weekly regional demand
regional_file = (
    processed_dir / "regional_demand_12_weeks.csv"
)

regional_demand.to_csv(
    regional_file,
    index=False
)

# Verify exported files
assert allocation_file.is_file()
assert regional_file.is_file()

print("SUCCESS: Regional demand allocation exported!")
print("\nFiles created:")
print(allocation_file)
print(regional_file)

print("\nRecords:")
print("Market allocation:", len(market_allocation))
print("Regional weekly demand:", len(regional_demand))

print("\nTotal regional demand:",
      regional_demand["Demand_Units"].sum())

print("Validation checks: PASSED")


SUCCESS: Regional demand allocation exported!

Files created:
/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/assumptions/market_allocation.csv
/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/processed/regional_demand_12_weeks.csv

Records:
Market allocation: 2
Regional weekly demand: 24

Total regional demand: 6180
Validation checks: PASSED



## Step 9 — Complete Supply Network Validation

### Objective
Perform a final data-integrity audit across
all supply network input datasets.

### Validation Areas
1. Dataset completeness
2. Product and component relationships
3. Supplier and plant relationships
4. Transportation network connectivity
5. Distribution center inventory capacity
6. Regional demand reconciliation
7. Initial inventory and lead-time readiness

### Important Distinction
Passing these checks establishes data integrity
and basic planning consistency.

Full operational feasibility will be determined
by the time-phased baseline and optimization models
in subsequent notebooks.


In [24]:

# STEP 9: Reload all network datasets

import pandas as pd
from pathlib import Path

repo_root = Path.cwd()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

files = {
    "bom": "data/assumptions/bom.csv",
    "suppliers": "data/assumptions/suppliers.csv",
    "plants": "data/assumptions/plants.csv",
    "dcs": "data/assumptions/distribution_centers.csv",
    "transport": "data/assumptions/transport_lanes.csv",
    "inventory": "data/assumptions/initial_inventory.csv",
    "allocation": "data/assumptions/market_allocation.csv",
    "demand": "data/processed/demand_plan_12_weeks.csv",
    "regional": "data/processed/regional_demand_12_weeks.csv"
}

# Verify that every file exists
missing = [
    name for name, path in files.items()
    if not (repo_root / path).is_file()
]

assert not missing, f"Missing datasets: {missing}"

# Load all datasets
network = {
    name: pd.read_csv(repo_root / path)
    for name, path in files.items()
}

print("SUPPLY NETWORK DATA AUDIT")
print("=" * 45)

for name, df in network.items():
    print(f"{name}: {len(df)} records")

print("\nAll 9 datasets loaded successfully!")


SUPPLY NETWORK DATA AUDIT
bom: 3 records
suppliers: 6 records
plants: 1 records
dcs: 2 records
transport: 4 records
inventory: 6 records
allocation: 2 records
demand: 12 records
regional: 24 records

All 9 datasets loaded successfully!


In [25]:
# STEP 9a: Validate network relationships

import numpy as np

b = network["bom"]
s = network["suppliers"]
p = network["plants"]
dc = network["dcs"]
t = network["transport"]
inv = network["inventory"]
a = network["allocation"]
d = network["demand"]
r = network["regional"]

# No missing values in required datasets
for name, df in network.items():
    assert df.notna().all().all(), (
        f"Missing values in {name}"
    )

# Product relationships
assert set(b["SKU"]) == set(d["SKU"])
assert set(p["SKU"]) == set(d["SKU"])
assert set(dc["SKU"]) == set(d["SKU"])

# Supplier-component relationships
assert set(b["Component_ID"]).issubset(
    set(s["Component_ID"])
)

assert not s.duplicated(
    ["Supplier_ID", "Component_ID"]
).any()

# Transportation relationships
assert set(t["Origin_Plant_ID"]).issubset(
    set(p["Plant_ID"])
)

assert set(t["Destination_DC_ID"]) == set(
    dc["DC_ID"]
)

# Regional allocation relationships
assert set(a["DC_ID"]) == set(dc["DC_ID"])

assert set(
    r[["Region", "DC_ID"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
) == set(
    a[["Region", "DC_ID"]]
    .itertuples(index=False, name=None)
)

assert np.isclose(a["Demand_Share"].sum(), 1.0)

# Demand reconciliation
weekly_original = d.groupby(
    "Week"
)["Demand_Units"].sum()

weekly_regional = r.groupby(
    "Week"
)["Demand_Units"].sum()

assert weekly_original.index.equals(
    weekly_regional.index
)

assert np.allclose(
    weekly_original.to_numpy(),
    weekly_regional.to_numpy()
)

# Initial inventory references
valid_locations = (
    set(p["Plant_ID"]) | set(dc["DC_ID"])
)

assert set(inv["Location_ID"]).issubset(
    valid_locations
)

valid_items = (
    set(b["Component_ID"]) | set(d["SKU"])
)

assert set(inv["Item_ID"]).issubset(valid_items)

# Numerical integrity
assert s["Weekly_Capacity_Units"].gt(0).all()
assert p["Weekly_Capacity_Units"].gt(0).all()
assert dc["Storage_Capacity_Units"].gt(0).all()
assert t["Weekly_Lane_Capacity_Units"].gt(0).all()
assert inv["Beginning_Inventory_Units"].ge(0).all()

print("SUCCESS: All network relationships validated!")
print("Product relationships: PASSED")
print("Supplier relationships: PASSED")
print("Transportation relationships: PASSED")
print("Inventory references: PASSED")
print("Regional demand reconciliation: PASSED")


SUCCESS: All network relationships validated!
Product relationships: PASSED
Supplier relationships: PASSED
Transportation relationships: PASSED
Inventory references: PASSED
Regional demand reconciliation: PASSED


In [27]:

# STEP 9b: Export network validation summary

import pandas as pd
from pathlib import Path

# Create validation summary
audit = pd.DataFrame([
    ["Network Datasets", len(network), "PASS"],
    ["Products", d["SKU"].nunique(), "PASS"],
    ["Component Groups", len(b), "PASS"],
    ["Suppliers", s["Supplier_ID"].nunique(), "PASS"],
    ["Manufacturing Plants", len(p), "PASS"],
    ["Distribution Centers", len(dc), "PASS"],
    ["Transport Lanes", len(t), "PASS"],
    ["Planning Weeks", d["Week"].nunique(), "PASS"],
    ["Regional Demand Records", len(r), "PASS"],
    ["Total Planned Demand", d["Demand_Units"].sum(), "PASS"],
    ["Week 1 Inventory Coverage", "Verified", "PASS"],
    ["Cross-Table Relationships", "Verified", "PASS"],
    ["Time-Phased Feasibility", "Not Yet Tested", "PENDING"]
], columns=[
    "Validation_Metric",
    "Value",
    "Status"
])

# Create output directory
output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

# Export audit results
audit_file = (
    output_dir / "network_validation_summary.csv"
)

audit.to_csv(audit_file, index=False)

# Verify export
assert audit_file.is_file()
assert len(audit) == 13

print("SUCCESS: Network validation summary exported!")
print("File:", audit_file)
print("Validation records:", len(audit))

print("\nVALIDATION SUMMARY:")
display(audit)


SUCCESS: Network validation summary exported!
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables/network_validation_summary.csv
Validation records: 13

VALIDATION SUMMARY:


,Validation_Metric,Value,Status
0,Network Datasets,9,PASS
1,Products,1,PASS
2,Component Groups,3,PASS
3,Suppliers,2,PASS
4,Manufacturing Plants,1,PASS
5,Distribution Centers,2,PASS
6,Transport Lanes,4,PASS
7,Planning Weeks,12,PASS
8,Regional Demand Records,24,PASS
9,Total Planned Demand,6180.0,PASS


In [28]:
# STEP 9c: Export network validation summary

audit = pd.DataFrame([
    ["Network Datasets", len(network), "PASS"],
    ["Products", d["SKU"].nunique(), "PASS"],
    ["Component Groups", len(b), "PASS"],
    ["Suppliers", s["Supplier_ID"].nunique(), "PASS"],
    ["Manufacturing Plants", len(p), "PASS"],
    ["Distribution Centers", len(dc), "PASS"],
    ["Transport Lanes", len(t), "PASS"],
    ["Planning Weeks", d["Week"].nunique(), "PASS"],
    ["Regional Demand Records", len(r), "PASS"],
    ["Total Planned Demand", d["Demand_Units"].sum(), "PASS"],
    ["Week 1 Inventory Coverage", "Verified", "PASS"],
    ["Cross-Table Relationships", "Verified", "PASS"],
    ["Time-Phased Feasibility", "Not Yet Tested", "PENDING"]
], columns=[
    "Validation_Metric",
    "Value",
    "Status"
])

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

audit_file = output_dir / "network_validation_summary.csv"

audit.to_csv(audit_file, index=False)

print("Network validation summary exported!")
print("File:", audit_file)
display(audit)


Network validation summary exported!
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables/network_validation_summary.csv


,Validation_Metric,Value,Status
0,Network Datasets,9,PASS
1,Products,1,PASS
2,Component Groups,3,PASS
3,Suppliers,2,PASS
4,Manufacturing Plants,1,PASS
5,Distribution Centers,2,PASS
6,Transport Lanes,4,PASS
7,Planning Weeks,12,PASS
8,Regional Demand Records,24,PASS
9,Total Planned Demand,6180.0,PASS
